# 📊 Dataset Generation & Exploration
### AI Interview Simulator — ML Pipeline

This notebook generates synthetic training datasets for all 5 specialized ML models and provides comprehensive visualizations for data understanding and quality assurance.

**Datasets:**
1. **Answer Quality** — Question + Answer → Score (0-10)
2. **Communication** — Text → Clarity / Fluency / Structure (0-5)
3. **STAR Analyzer** — Behavioral Answer → S/T/A/R scores (0-5)
4. **Code Evaluator** — Source Code → Quality / Efficiency / Style (0-5)
5. **Meta Scorer** — 16-feature vector → Final Score (0-10)

In [ ]:
import sys, os
sys.path.insert(0, os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == 'ml' else os.getcwd())

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from collections import Counter

# Styling
plt.style.use('dark_background')
sns.set_palette('viridis')
BRAND = '#6366f1'
COLORS = ['#6366f1', '#22c55e', '#eab308', '#ef4444', '#06b6d4', '#ec4899']

print('✓ Libraries loaded')

## 1. Generate All Datasets

In [ ]:
from ml.dataset import (
    generate_answer_quality_data,
    generate_communication_data,
    generate_star_data,
    generate_code_eval_data,
    generate_meta_scorer_data,
    generate_all
)

N = 2000
OUTPUT_DIR = os.path.join('..', 'data', 'ml_training')

# Generate and save
generate_all(OUTPUT_DIR, n_per_dataset=N)

# Load as DataFrames
df_aq = pd.DataFrame(generate_answer_quality_data(N))
df_comm = pd.DataFrame(generate_communication_data(N))
df_star = pd.DataFrame(generate_star_data(N))
df_code = pd.DataFrame(generate_code_eval_data(1500))
df_meta = pd.DataFrame(generate_meta_scorer_data(3000))

datasets = {
    'Answer Quality': df_aq,
    'Communication': df_comm,
    'STAR Analyzer': df_star,
    'Code Evaluator': df_code,
    'Meta Scorer': df_meta,
}

print('\n📋 Dataset Sizes:')
for name, df in datasets.items():
    print(f'   {name:20s} → {len(df):,} samples, {len(df.columns)} features')

## 2. Answer Quality Dataset
**Input:** Question + Answer (text pair)  
**Target:** Score 0-10 (regression)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle('Answer Quality Dataset', fontsize=14, fontweight='bold', color='white')

# Score distribution
axes[0].hist(df_aq['score'], bins=30, color=BRAND, edgecolor='white', alpha=0.8)
axes[0].set_xlabel('Score', fontsize=10)
axes[0].set_ylabel('Count', fontsize=10)
axes[0].set_title('Score Distribution', fontsize=11)
axes[0].axvline(df_aq['score'].mean(), color='#ef4444', linestyle='--', label=f"Mean: {df_aq['score'].mean():.2f}")
axes[0].legend(fontsize=8)

# Answer length distribution
df_aq['answer_len'] = df_aq['answer'].str.len()
axes[1].hist(df_aq['answer_len'], bins=30, color='#22c55e', edgecolor='white', alpha=0.8)
axes[1].set_xlabel('Answer Length (chars)', fontsize=10)
axes[1].set_ylabel('Count', fontsize=10)
axes[1].set_title('Answer Length Distribution', fontsize=11)

# Score vs answer length scatter
axes[2].scatter(df_aq['answer_len'], df_aq['score'], alpha=0.3, s=8, c=df_aq['score'], cmap='viridis')
axes[2].set_xlabel('Answer Length', fontsize=10)
axes[2].set_ylabel('Score', fontsize=10)
axes[2].set_title('Score vs Length', fontsize=11)

plt.tight_layout()
plt.show()

print(df_aq[['score', 'answer_len']].describe().round(2))

## 3. Communication Clarity Dataset
**Input:** Answer text  
**Targets:** Clarity, Fluency, Structure (0-5 each)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle('Communication Clarity Dataset', fontsize=14, fontweight='bold', color='white')

for idx, (col, color) in enumerate(zip(['clarity', 'fluency', 'structure'], COLORS[:3])):
    axes[idx].hist(df_comm[col], bins=25, color=color, edgecolor='white', alpha=0.8)
    axes[idx].set_xlabel(col.title(), fontsize=10)
    axes[idx].set_ylabel('Count', fontsize=10)
    axes[idx].set_title(f'{col.title()} Distribution', fontsize=11)
    axes[idx].axvline(df_comm[col].mean(), color='#ef4444', linestyle='--', label=f"μ={df_comm[col].mean():.2f}")
    axes[idx].legend(fontsize=8)

plt.tight_layout()
plt.show()

# Correlation matrix
fig, ax = plt.subplots(figsize=(5, 4))
corr = df_comm[['clarity', 'fluency', 'structure']].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='magma', ax=ax, square=True, linewidths=1)
ax.set_title('Communication Dimensions Correlation', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. STAR Behavioral Analyzer Dataset
**Input:** Behavioral answer text  
**Targets:** Situation, Task, Action, Result scores (0-5 each)

In [ ]:
star_cols = ['situation_score', 'task_score', 'action_score', 'result_score']
star_labels = ['Situation', 'Task', 'Action', 'Result']
star_colors = ['#3b82f6', '#8b5cf6', '#22c55e', '#f59e0b']

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
fig.suptitle('STAR Behavioral Analyzer Dataset', fontsize=14, fontweight='bold', color='white')

for idx, (col, label, color) in enumerate(zip(star_cols, star_labels, star_colors)):
    ax = axes[idx // 2][idx % 2]
    ax.hist(df_star[col], bins=25, color=color, edgecolor='white', alpha=0.8)
    ax.set_xlabel(label, fontsize=10)
    ax.set_ylabel('Count', fontsize=10)
    ax.set_title(f'{label} Score Distribution', fontsize=11)
    ax.axvline(df_star[col].mean(), color='#ef4444', linestyle='--', label=f"μ={df_star[col].mean():.2f}")
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

# Radar chart for average STAR scores
angles = np.linspace(0, 2 * np.pi, 4, endpoint=False).tolist()
angles += angles[:1]  # close polygon
means = [df_star[c].mean() for c in star_cols]
means += means[:1]

fig, ax = plt.subplots(figsize=(5, 5), subplot_kw={'projection': 'polar'})
ax.fill(angles, means, alpha=0.25, color=BRAND)
ax.plot(angles, means, 'o-', color=BRAND, linewidth=2)
ax.set_xticks(angles[:-1])
ax.set_xticklabels(star_labels, fontsize=10, fontweight='bold')
ax.set_ylim(0, 5)
ax.set_title('Average STAR Profile', fontsize=12, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

## 5. Code Evaluator Dataset
**Input:** Source code (Python)  
**Targets:** Quality, Efficiency, Style (0-5 each)

In [ ]:
code_cols = ['quality', 'efficiency', 'style']
code_colors = ['#06b6d4', '#22c55e', '#ec4899']

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle('Code Evaluator Dataset', fontsize=14, fontweight='bold', color='white')

for idx, (col, color) in enumerate(zip(code_cols, code_colors)):
    axes[idx].hist(df_code[col], bins=25, color=color, edgecolor='white', alpha=0.8)
    axes[idx].set_xlabel(col.title(), fontsize=10)
    axes[idx].set_ylabel('Count', fontsize=10)
    axes[idx].set_title(f'{col.title()} Distribution', fontsize=11)
    axes[idx].axvline(df_code[col].mean(), color='#ef4444', linestyle='--', label=f"μ={df_code[col].mean():.2f}")
    axes[idx].legend(fontsize=8)

plt.tight_layout()
plt.show()

# Code length analysis
df_code['code_len'] = df_code['code'].str.len()
fig, ax = plt.subplots(figsize=(8, 4))
ax.scatter(df_code['code_len'], df_code['quality'], alpha=0.4, s=15, c=df_code['efficiency'], cmap='viridis')
ax.set_xlabel('Code Length (chars)', fontsize=10)
ax.set_ylabel('Quality Score', fontsize=10)
ax.set_title('Code Quality vs Length (colored by Efficiency)', fontsize=11, fontweight='bold')
plt.colorbar(ax.collections[0], label='Efficiency')
plt.tight_layout()
plt.show()

## 6. Meta Scorer Dataset
**Input:** 16-dimensional feature vector from all models  
**Target:** Final aggregated score (0-10)

In [ ]:
fig = plt.figure(figsize=(16, 10))
gs = gridspec.GridSpec(2, 3, figure=fig)
fig.suptitle('Meta Scorer Dataset — Feature Analysis', fontsize=14, fontweight='bold', color='white')

# Final score distribution
ax1 = fig.add_subplot(gs[0, 0])
ax1.hist(df_meta['final_score'], bins=30, color=BRAND, edgecolor='white', alpha=0.8)
ax1.axvline(df_meta['final_score'].mean(), color='#ef4444', linestyle='--', label=f"μ={df_meta['final_score'].mean():.2f}")
ax1.set_title('Final Score Distribution', fontsize=11)
ax1.legend(fontsize=8)

# Top correlations with final score
ax2 = fig.add_subplot(gs[0, 1:])
feat_cols = [c for c in df_meta.columns if c != 'final_score']
correlations = df_meta[feat_cols].corrwith(df_meta['final_score']).sort_values(ascending=True)
colors_bar = [BRAND if v > 0 else '#ef4444' for v in correlations]
correlations.plot(kind='barh', ax=ax2, color=colors_bar, edgecolor='white')
ax2.set_title('Feature Correlation with Final Score', fontsize=11)
ax2.set_xlabel('Pearson Correlation', fontsize=10)

# Feature distributions (top 4)
top_feats = correlations.abs().nlargest(4).index.tolist()
for idx, feat in enumerate(top_feats):
    ax = fig.add_subplot(gs[1, idx] if idx < 3 else gs[1, 2])
    ax.scatter(df_meta[feat], df_meta['final_score'], alpha=0.15, s=5, c=BRAND)
    ax.set_xlabel(feat.replace('_', ' ').title(), fontsize=9)
    ax.set_ylabel('Final Score', fontsize=9)
    ax.set_title(f'r = {df_meta[feat].corr(df_meta["final_score"]):.3f}', fontsize=10)

plt.tight_layout()
plt.show()

In [ ]:
# Full correlation heatmap for meta scorer
fig, ax = plt.subplots(figsize=(14, 10))
corr_full = df_meta.corr()
mask = np.triu(np.ones_like(corr_full, dtype=bool))
sns.heatmap(corr_full, mask=mask, annot=True, fmt='.2f', cmap='coolwarm', ax=ax,
            square=True, linewidths=0.5, center=0, vmin=-1, vmax=1,
            annot_kws={'fontsize': 7})
ax.set_title('Meta Scorer — Full Feature Correlation Matrix', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 7. Cross-Dataset Summary Statistics

In [ ]:
summary = pd.DataFrame({
    'Dataset': ['Answer Quality', 'Communication', 'STAR Analyzer', 'Code Evaluator', 'Meta Scorer'],
    'Samples': [len(df_aq), len(df_comm), len(df_star), len(df_code), len(df_meta)],
    'Features': [len(df_aq.columns), len(df_comm.columns), len(df_star.columns), len(df_code.columns), len(df_meta.columns)],
    'Target Columns': ['score', 'clarity, fluency, structure', 'S, T, A, R', 'quality, efficiency, style', 'final_score'],
    'Target Range': ['0-10', '0-5', '0-5', '0-5', '0-10'],
})

print('\n📊 Dataset Summary')
print('=' * 90)
print(summary.to_string(index=False))
print('=' * 90)
print(f'\nTotal samples across all datasets: {summary["Samples"].sum():,}')